## Análises de Soft Skills e Normalização de Vagas
Notebook com todas as análises solicitadas:
1. Amostras de soft skills (criatividade, comunicação, adaptabilidade)
2. Amostra aleatória para varredura manual
3. Análise de vagas juniores
4. Identificação de vagas não normalizadas

In [1]:
import pandas as pd
import re
import numpy as np
from pathlib import Path

### Carrega Dados Processados

In [2]:
# Carregar dados do scp_02
df = pd.read_csv('data/scp_02.csv')

print(f"Total de vagas carregadas: {len(df):,}")
print(f"\nColunas disponíveis: {list(df.columns)}")
print(f"\nPreview dos dados:")
df.head()

Total de vagas carregadas: 15,514

Colunas disponíveis: ['Cargo', 'Skills', 'Area', 'Tipo', 'Nivel', 'Local Cidade', 'Local Estado', 'Local Tipo', 'Salario']

Preview dos dados:


,Cargo,Skills,Area,Tipo,Nivel,Local Cidade,Local Estado,Local Tipo,Salario
0,application support analyst - remote work,"At BairesDev®, we've been leading the way in t...",Desenvolvimento de Software,Cargo técnico,n/f,São Bernardo do Campo,São Paulo,nacional,n/f
1,tech lead - - ai - first product engineering,About the RoleLivvie is looking for a hands-on...,Projetos e Gerenciamento,Cargo gestão,Nível Liderança,Curitiba,Paraná,nacional,n/f
2,tech lead - - ai - first product engineering,About the RoleLivvie is looking for a hands-on...,Projetos e Gerenciamento,Cargo gestão,Nível Liderança,Trabalho remoto,Trabalho remoto,remoto,n/f
3,backend javascript engineer - remote work,"At BairesDev®, we've been leading the way in t...",Desenvolvimento de Software,Cargo técnico,n/f,Salvador,Bahia,nacional,n/f
4,"full stack / backend engineer , ai focus","Full Stack / Backend Engineer, AI FocusContrac...",Desenvolvimento de Software,Cargo técnico,n/f,São Paulo,São Paulo,nacional,n/f


---
## 1. AMOSTRAS DE SOFT SKILLS (50 vagas cada)

In [3]:
def buscar_softskill(df, termo, n_amostras=50):
    """
    Busca vagas que mencionam determinada soft skill e retorna amostra
    
    Parâmetros:
    - df: amostraFrame com as vagas
    - termo: 'criatividade', 'comunicacao' ou 'adaptabilidade'
    - n_amostras: número de vagas a retornar
    """
    # Normalizar texto
    df_copy = df.copy()
    df_copy['Skills_lower'] = df_copy['Skills'].fillna('').str.lower()
    
    # Dicionário com variações de cada soft skill
    variacoes = {
        'criatividade': [
            'criativ', 'creative', 
            'pensamento criativo', 'solucoes criativas'
        ],
        'comunicacao': [
            'comunicaç', 'comunicac', 'comunicar', 'communication', 
            'comunicacao eficaz', 'boa comunicacao', 'habilidade de comunicacao',
            'comunicacao oral', 'comunicacao escrita'
        ],
        'adaptabilidade': [
            'adaptab', 'flexib', 'adaptable', 'flexibility', 
            'facilidade de adaptacao', 'versatil', 'versatilidade'
        ]
    }
    
    # Buscar vagas que contêm alguma variação
    pattern = '|'.join(variacoes.get(termo, [termo]))
    vagas_encontradas = df_copy[df_copy['Skills_lower'].str.contains(pattern, na=False, regex=True)]
    
    print(f"\n{'='*60}")
    print(f"SOFT SKILL: {termo.upper()}")
    print(f"{'='*60}")
    print(f"Total de vagas encontradas: {len(vagas_encontradas):,}")
    
    if len(vagas_encontradas) == 0:
        print("⚠️  Nenhuma vaga encontrada com essa soft skill")
        return pd.amostraFrame()
    
    # Amostra aleatória estratificada por área (se possível)
    if len(vagas_encontradas) > n_amostras:
        # Estratifica por área (compatível com pandas 2.x)
        try:
            n_areas = vagas_encontradas['Area'].nunique()
            n_per_area = max(1, n_amostras // n_areas)
            parts = [
                g.sample(min(len(g), n_per_area), random_state=42)
                for _, g in vagas_encontradas.groupby('Area', group_keys=False)
            ]
            amostra = pd.concat(parts).head(n_amostras)
        except Exception:
            amostra = vagas_encontradas.sample(n=n_amostras, random_state=42)
    else:
        amostra = vagas_encontradas
        print(f"⚠️  Retornando todas as {len(amostra)} vagas (menos que {n_amostras})")
    
    # Estatísticas
    print(f"\nAmostra selecionada: {len(amostra)} vagas")
    print(f"\nDistribuição por Área:")
    print(amostra['Area'].value_counts())
    print(f"\nDistribuição por Nível:")
    print(amostra['Nivel'].value_counts())
    
    return amostra[['Cargo', 'Area', 'Tipo', 'Nivel', 'Skills', 'Local Cidade', 'Local Estado', 'Salario']]

In [4]:
# Executar buscas para cada soft skill
amostra_criatividade = buscar_softskill(df, 'criatividade', 50)
amostra_comunicacao = buscar_softskill(df, 'comunicacao', 50)
amostra_adaptabilidade = buscar_softskill(df, 'adaptabilidade', 50)

# Salvar amostras
Path('amostra').mkdir(exist_ok=True)

if not amostra_criatividade.empty:
    amostra_criatividade.to_excel('amostra/amostra_criatividade.xlsx', index=False)
    print("\n✅ Salvo: amostra/amostra_criatividade.xlsx")

if not amostra_comunicacao.empty:
    amostra_comunicacao.to_excel('amostra/amostra_comunicacao.xlsx', index=False)
    print("✅ Salvo: amostra/amostra_comunicacao.xlsx")

if not amostra_adaptabilidade.empty:
    amostra_adaptabilidade.to_excel('amostra/amostra_adaptabilidade.xlsx', index=False)
    print("✅ Salvo: amostra/amostra_adaptabilidade.xlsx")


SOFT SKILL: CRIATIVIDADE
Total de vagas encontradas: 1,433

Amostra selecionada: 49 vagas

Distribuição por Área:
Area
Automação de Processos e Sistemas    7
Dados e Inteligência Artificial      7
Desenvolvimento de Software          7
Infraestrutura e Suporte             7
Projetos e Gerenciamento             7
Segurança da Informação              7
Testes e Controle de Qualidade       7
Name: count, dtype: int64

Distribuição por Nível:
Nivel
Nível Pleno / Sênior    24
Nível Júnior            12
n/f                      9
Nível Liderança          4
Name: count, dtype: int64



SOFT SKILL: COMUNICACAO
Total de vagas encontradas: 7,082

Amostra selecionada: 49 vagas

Distribuição por Área:
Area
Automação de Processos e Sistemas    7
Dados e Inteligência Artificial      7
Desenvolvimento de Software          7
Infraestrutura e Suporte             7
Projetos e Gerenciamento             7
Segurança da Informação              7
Testes e Controle de Qualidade       7
Name: count, dtype: int64

Distribuição por Nível:
Nivel
Nível Pleno / Sênior    25
n/f                     12
Nível Júnior             9
Nível Liderança          3
Name: count, dtype: int64



SOFT SKILL: ADAPTABILIDADE
Total de vagas encontradas: 2,020

Amostra selecionada: 49 vagas

Distribuição por Área:
Area
Automação de Processos e Sistemas    7
Dados e Inteligência Artificial      7
Desenvolvimento de Software          7
Infraestrutura e Suporte             7
Projetos e Gerenciamento             7
Segurança da Informação              7
Testes e Controle de Qualidade       7
Name: count, dtype: int64

Distribuição por Nível:
Nivel
Nível Pleno / Sênior    21
n/f                     21
Nível Júnior             4
Nível Liderança          3
Name: count, dtype: int64



✅ Salvo: amostra/amostra_criatividade.xlsx
✅ Salvo: amostra/amostra_comunicacao.xlsx
✅ Salvo: amostra/amostra_adaptabilidade.xlsx


---
## 2. AMOSTRA ALEATÓRIA PARA VARREDURA MANUAL (200 vagas)

In [5]:
print(f"\n{'='*60}")
print("AMOSTRA ALEATÓRIA PARA VARREDURA MANUAL")
print(f"{'='*60}")

# Amostra estratificada por área (proporcional)
n_total = 200

# Calcular proporção por área
proporcao_area = df['Area'].value_counts(normalize=True)
print("\nDistribuição original por área:")
print(proporcao_area)

# Criar amostra estratificada
parts = [
    g.sample(min(len(g), max(1, int(n_total * len(g) / len(df)))), random_state=42)
    for _, g in df.groupby('Area', group_keys=False)
]
amostra_geral = pd.concat(parts).head(n_total)

print(f"\n✅ Amostra de {len(amostra_geral)} vagas selecionada")
print("\nDistribuição na amostra:")
print(amostra_geral['Area'].value_counts())

# Selecionar colunas relevantes
colunas_relevantes = ['Cargo', 'Area', 'Tipo', 'Nivel', 'Skills', 
                      'Local Cidade', 'Local Estado', 'Local Tipo', 'Salario']

amostra_exportar = amostra_geral[colunas_relevantes].copy()

# Adicionar coluna para marcação manual de soft skills
amostra_exportar['Soft_Skills_Identificadas'] = ''
amostra_exportar['Observacoes'] = ''

# Salvar
amostra_exportar.to_excel('amostra/amostra_varredura_manual_200.xlsx', index=False)
print("\n✅ Salvo: amostra/amostra_varredura_manual_200.xlsx")


AMOSTRA ALEATÓRIA PARA VARREDURA MANUAL

Distribuição original por área:
Area
Desenvolvimento de Software          0.306691
Projetos e Gerenciamento             0.215870
Infraestrutura e Suporte             0.205492
Dados e Inteligência Artificial      0.158180
Testes e Controle de Qualidade       0.071484
Segurança da Informação              0.024236
Automação de Processos e Sistemas    0.018048
Name: proportion, dtype: float64

✅ Amostra de 197 vagas selecionada

Distribuição na amostra:
Area
Desenvolvimento de Software          61
Projetos e Gerenciamento             43
Infraestrutura e Suporte             41
Dados e Inteligência Artificial      31
Testes e Controle de Qualidade       14
Segurança da Informação               4
Automação de Processos e Sistemas     3
Name: count, dtype: int64

✅ Salvo: amostra/amostra_varredura_manual_200.xlsx


In [6]:
df_automacao = df[df['Area'].str.contains('Automacao|Automação', case=False, na=False)]

print("Total de vagas em Automação:", len(df_automacao))

df_automacao_sample = df_automacao.sample(
    min(100, len(df_automacao)),
    random_state=42
)

df_automacao_sample.to_excel("amostra/amostra_automacao_100.xlsx", index=False)
print("\n✅ Salvo: amostra/amostra_automacao_100.xlsx")

display(df_automacao_sample)

Total de vagas em Automação: 280

✅ Salvo: amostra/amostra_automacao_100.xlsx


,Cargo,Skills,Area,Tipo,Nivel,Local Cidade,Local Estado,Local Tipo,Salario
2420,estagio em controle e automacao - engenharia...,Estágio em Controle e Automação | Porto Alegre...,Automação de Processos e Sistemas,Cargo técnico,Nível Júnior,Porto Alegre,Rio Grande do Sul,nacional,n/f
6826,senior operations automation analyst,Como o time de Processos e Melhoria Contínua c...,Automação de Processos e Sistemas,Cargo técnico,Nível Pleno / Sênior,Curitiba,Paraná,nacional,n/f
13512,assistente administrativo ( tbsa - tecnolog...,Empresa de Soluções em Construções. Pavimentaç...,Automação de Processos e Sistemas,Cargo inicial,Nível Júnior,n/f,n/f,n/f,n/f
13957,analista de engenharia de automacao e controle...,Quer contribuir com o Ecossistema das Lojas Re...,Automação de Processos e Sistemas,Cargo técnico,Nível Pleno / Sênior,Cabreúva,São Paulo,nacional,n/f
9316,orcamentista tecnico - paineis eletricos e a...,Orçamentista Técnico – Painéis Elétricos e Aut...,Automação de Processos e Sistemas,Cargo técnico,Nível Júnior,São José dos Campos,São Paulo,nacional,n/f
...,...,...,...,...,...,...,...,...,...
2405,tecnico de automacao ii,Empresa de grande porte no segmneto de auto pe...,Automação de Processos e Sistemas,Cargo técnico,Nível Júnior,Betim,Minas Gerais,nacional,n/f
13915,analista de automacao de testes pleno,A Cyber Solutions é uma empresa de tecnologia ...,Automação de Processos e Sistemas,Cargo técnico,Nível Pleno / Sênior,Barueri,São Paulo,nacional,n/f
4101,consultor de automacao e instrumentacao,Há 16 anos trabalhamos para resolver os desafi...,Automação de Processos e Sistemas,Cargo técnico,Nível Pleno / Sênior,Belo Horizonte,Minas Gerais,nacional,n/f
10874,analista de rpa senior,Somos a única solução completa para o mercado ...,Automação de Processos e Sistemas,Cargo técnico,Nível Pleno / Sênior,São Paulo,São Paulo,nacional,n/f


---
## 3. ANÁLISE DE VAGAS JUNIORES

### 3.1 - Percentual de cada área dentro das vagas juniores

### 3.2 - Diferença salarial entre níveis

In [7]:
print(f"\n{'='*60}")
print("ANÁLISE SALARIAL POR NÍVEL")
print(f"{'='*60}")

def extrair_salario_medio(faixa):
    """
    Converte faixas salariais em valor médio
    Exemplos:
    - '2000-4000' -> 3000
    - '5000' -> 5000
    - 'R$ 3.000 - R$ 5.000' -> 4000
    """
    if pd.isna(faixa) or faixa in ['n/e', 'n/f', '']:
        return None
    
    # Remover caracteres não numéricos exceto dígitos e hífen
    faixa_limpa = re.sub(r'[^0-9-]', '', str(faixa))
    
    # Extrair números
    numeros = re.findall(r'\d+', faixa_limpa)
    
    if len(numeros) >= 2:
        # Pegar os dois primeiros números (min e max da faixa)
        return (int(numeros[0]) + int(numeros[1])) / 2
    elif len(numeros) == 1:
        return int(numeros[0])
    
    return None

# Filtrar vagas com salário informado
vagas_com_salario = df[~df['Salario'].isin(['n/e', 'n/f', ''])].copy()

print(f"\nVagas com salário informado: {len(vagas_com_salario):,} ({len(vagas_com_salario)/len(df)*100:.1f}%)")

# Converter para valor numérico
vagas_com_salario['Salario_Medio'] = pd.to_numeric(vagas_com_salario['Salario'].apply(extrair_salario_medio), errors='coerce')
vagas_com_salario = vagas_com_salario[vagas_com_salario['Salario_Medio'].notna()]

print(f"Vagas com salário convertido com sucesso: {len(vagas_com_salario):,}")

# Análise por nível
salario_por_nivel = vagas_com_salario.groupby('Nivel')['Salario_Medio'].agg([
    ('Média', 'mean'),
    ('Mediana', 'median'),
    ('Mínimo', 'min'),
    ('Máximo', 'max'),
    ('Quantidade', 'count')
])

salario_por_nivel = salario_por_nivel.round(2)
salario_por_nivel = salario_por_nivel.sort_values('Média')

print("\n" + salario_por_nivel.to_string())

# Calcular diferenças percentuais
print("\n" + "="*60)
print("DIFERENÇAS SALARIAIS ENTRE NÍVEIS")
print("="*60)

niveis_ordenados = ['estagio', 'junior', 'pleno', 'senior', 'especialista', 'lead', 'gerente', 'diretor']
niveis_presentes = [n for n in niveis_ordenados if n in salario_por_nivel.index]

for i in range(len(niveis_presentes) - 1):
    nivel_atual = niveis_presentes[i]
    nivel_proximo = niveis_presentes[i + 1]
    
    if nivel_atual in salario_por_nivel.index and nivel_proximo in salario_por_nivel.index:
        media_atual = salario_por_nivel.loc[nivel_atual, 'Média']
        media_proxima = salario_por_nivel.loc[nivel_proximo, 'Média']
        
        diff_absoluta = media_proxima - media_atual
        diff_percentual = (diff_absoluta / media_atual) * 100
        
        print(f"\n{nivel_atual.capitalize()} → {nivel_proximo.capitalize()}:")
        print(f"  Diferença: R$ {diff_absoluta:,.2f} (+{diff_percentual:.1f}%)")
        print(f"  De R$ {media_atual:,.2f} para R$ {media_proxima:,.2f}")

# Salvar análise
salario_por_nivel.to_excel('data/analise_salario_por_nivel.xlsx')
print("\n✅ Salvo: data/analise_salario_por_nivel.xlsx")


ANÁLISE SALARIAL POR NÍVEL

Vagas com salário informado: 0 (0.0%)
Vagas com salário convertido com sucesso: 0

Empty DataFrame
Columns: [Média, Mediana, Mínimo, Máximo, Quantidade]
Index: []

DIFERENÇAS SALARIAIS ENTRE NÍVEIS

✅ Salvo: data/analise_salario_por_nivel.xlsx


In [8]:
len(df['Cargo'].unique())

13135

---
## 4. VAGAS NÃO NORMALIZADAS (Amostra de 200)

In [9]:
print(f"\n{'='*60}")
print("IDENTIFICAÇÃO DE CARGOS NÃO NORMALIZADOS")
print(f"{'='*60}")

# Apenas cargos não normalizados
vagas_nao_normalizadas = df[df['Cargo'] == 'n/f'].copy()

print(f"\nTotal de cargos não normalizados: {len(vagas_nao_normalizadas):,}")
print(f"Percentual do total: {len(vagas_nao_normalizadas)/len(df)*100:.1f}%")

# Amostra de 200 vagas
if len(vagas_nao_normalizadas) > 200:
    amostra_nao_norm = vagas_nao_normalizadas.sample(n=200, random_state=42)
    print("\n✅ Amostra de 200 vagas selecionada aleatoriamente")
else:
    amostra_nao_norm = vagas_nao_normalizadas
    print(f"\n⚠️ Retornando todas as {len(amostra_nao_norm)} vagas")

# Exportar
colunas_export = [
    'Cargo', 'Area', 'Tipo', 'Nivel',
    'Local Cidade', 'Local Estado', 'Local Tipo',
    'Salario', 'Skills'
]

amostra_nao_norm[colunas_export].to_excel(
    'amostra/amostra_vagas_nao_normalizados.xlsx',
    index=False
)

print("\n✅ Salvo: amostra/amostra_vagas_nao_normalizados.xlsx")


IDENTIFICAÇÃO DE CARGOS NÃO NORMALIZADOS

Total de cargos não normalizados: 0
Percentual do total: 0.0%

⚠️ Retornando todas as 0 vagas

✅ Salvo: amostra/amostra_vagas_nao_normalizados.xlsx


---
## RESUMO DOS ARQUIVOS GERADOS

Todos os arquivos foram salvos na pasta `data/`:

1. **amostra_criatividade.xlsx** - 50 vagas que mencionam criatividade
2. **amostra_comunicacao.xlsx** - 50 vagas que mencionam comunicação
3. **amostra_adaptabilidade.xlsx** - 50 vagas que mencionam adaptabilidade
4. **amostra_varredura_manual_200.xlsx** - 200 vagas aleatórias para varredura manual de soft skills
5. **distribuicao_area_junior.xlsx** - Percentual de cada área em vagas juniores
6. **analise_salario_por_nivel.xlsx** - Diferenças salariais entre níveis
7. **amostra_vagas_nao_normalizadas.xlsx** - 200 vagas com problemas de normalização

In [10]:
# Verificar arquivos gerados
import os

arquivos_esperados = [
    'amostra_criatividade.xlsx',
    'amostra_comunicacao.xlsx',
    'amostra_adaptabilidade.xlsx',
    'amostra_varredura_manual_200.xlsx',
    'distribuicao_area_junior.xlsx',
    'analise_salario_por_nivel.xlsx',
    'amostra_vagas_nao_normalizadas.xlsx'
]

print("\n" + "="*60)
print("VERIFICAÇÃO DOS ARQUIVOS GERADOS")
print("="*60)

for arquivo in arquivos_esperados:
    caminho = f'data/{arquivo}'
    if os.path.exists(caminho):
        tamanho = os.path.getsize(caminho) / 1024  # em KB
        print(f"✅ {arquivo:<45} ({tamanho:.1f} KB)")
    else:
        print(f"❌ {arquivo:<45} (NÃO ENCONTRADO)")

print("\n" + "="*60)
print("ANÁLISES CONCLUÍDAS!")
print("="*60)


VERIFICAÇÃO DOS ARQUIVOS GERADOS
❌ amostra_criatividade.xlsx                     (NÃO ENCONTRADO)
❌ amostra_comunicacao.xlsx                      (NÃO ENCONTRADO)
❌ amostra_adaptabilidade.xlsx                   (NÃO ENCONTRADO)
❌ amostra_varredura_manual_200.xlsx             (NÃO ENCONTRADO)
❌ distribuicao_area_junior.xlsx                 (NÃO ENCONTRADO)
✅ analise_salario_por_nivel.xlsx                (4.8 KB)
❌ amostra_vagas_nao_normalizadas.xlsx           (NÃO ENCONTRADO)

ANÁLISES CONCLUÍDAS!
